# PoliticHeadlinES 2026 - Task 1

Cuaderno centrado **solo en Task 1** (texto).

Esta versión mantiene el mismo pipeline y las mismas señales del notebook original, pero cambia la ejecución para hacerla **más rápida y rentable**:

- precomputación por lotes de embeddings
- TF-IDF aplicado en bloque
- reutilización matricial de señales durante el autoajuste de pesos


# 1. Instalación de dependencias

Estas celdas preparan las librerías necesarias para embeddings y stopwords.


In [12]:
# Modulo de dependencias para evitar errores de importacion en v10_2
%pip -q install -U sentence-transformers
%pip -q install -q nltk


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [13]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\quiqu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# 2. Importaciones y configuración del experimento

Aquí se centralizan los nombres de archivos, parámetros del modelo, pesos iniciales y opciones de evaluación.

La lógica del ranker no cambia respecto al notebook base; solo se optimiza la forma de ejecutar la precomputación y la búsqueda de pesos.


In [14]:
from __future__ import annotations

import json
import math
import re
import string
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# Configuracion principal del experimento
METHOD = "bi_encoder_tfidf_word_overlap_fusion_task1_autoweights_optimized_execution"
DATA_DIR = Path("evaluation_phase_dataset")
TRAIN_CSV = DATA_DIR / "train_corpora.csv"
DEV_CSV = DATA_DIR / "PoliticHeadlinES_phase_2_test_public.csv"
OUTPUT_SUBMISSION = "results_evaluation.csv"

TITLE_COLS = [f"title_{i}" for i in range(1, 11)]
TOKENS_ALL = [f"t{i}" for i in range(1, 11)]
N_COLS = 10
SIGNAL_KEYS = ("bi_encoder", "tfidf", "overlap")

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
MODEL_MAX_SEQ_LENGTH = 512
LOCAL_MODELS_DIR = Path("models")
DOWNLOAD_MODEL_IF_MISSING = True

# Pesos iniciales de partida para el ranker y busqueda automatica
W_BI_ENCODER =  1
W_TFIDF = 0
W_OVERLAP = 0

#W_BI_ENCODER =  0.55
#W_TFIDF = 0.30
#W_OVERLAP = 0.15

AUTO_TUNE_WEIGHTS = True
WEIGHT_GRID_STEP = 0.05

NDCG_K = 10
ALPHA = 0.9
SAVE_METRICS_JSON = True

# Parametros de ejecucion optimizada
ENCODE_BATCH_SIZE = 64
SHOW_PROGRESS_BAR = True


def _weights_from_values(w_bi_encoder: float, w_tfidf: float, w_overlap: float) -> Dict[str, float]:
    return {
        "bi_encoder": w_bi_encoder,
        "tfidf": w_tfidf,
        "overlap": w_overlap,
    }


def _active_signal_keys(weights: Dict[str, float]) -> List[str]:
    return [key for key in SIGNAL_KEYS if weights.get(key, 0.0) > 0]


def _combine_weighted_components(components: Dict[str, np.ndarray], weights: Dict[str, float]) -> np.ndarray:
    active_keys = _active_signal_keys(weights)
    if not active_keys:
        raise ValueError("La suma de pesos debe ser > 0")

    combined = np.zeros_like(components[active_keys[0]], dtype=float)
    weight_sum = 0.0
    for key in active_keys:
        combined += weights[key] * components[key]
        weight_sum += weights[key]
    return combined / weight_sum


def resolve_model_path(model_name: str, local_models_dir: Path, download_if_missing: bool = True) -> str:
    model_dir = local_models_dir / model_name.split("/")[-1]
    if model_dir.exists():
        print(f"Usando modelo local: {model_dir}")
        return str(model_dir)

    if not download_if_missing:
        raise FileNotFoundError(
            f"No se encontro el modelo en local: {model_dir}. "
            "Activa DOWNLOAD_MODEL_IF_MISSING para permitir la descarga automatica."
        )

    print(f"Modelo no encontrado en local. Descargando desde Hugging Face en: {model_dir}")
    snapshot_download(repo_id=model_name, local_dir=str(model_dir), local_dir_use_symlinks=False)
    return str(model_dir)


## 3. Carga de datos

Se usa `train_public.csv` para ajustar TF-IDF y `dev_public.csv` para predecir y evaluar Task 1.


In [15]:
# Cargamos train para ajustar TF-IDF y dev para predecir/evaluar Task 1
train_df = pd.read_csv(TRAIN_CSV)
dev_df = pd.read_csv(DEV_CSV)

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)


Train: (16030, 14)
Dev: (4912, 13)


# 4. Utilidades base y métricas oficiales

Primero se definen funciones auxiliares para:
- obtener el texto fuente y los títulos candidatos,
- parsear rankings,
- calcular la métrica oficial `PA-NDCG`,
- calcular cobertura y puntuación final del cuaderno.


In [16]:
# Utilidades generales para parsear rankings y calcular metricas

def tokens(x: Any) -> List[str]:
    if x is None:
        return []
    s = str(x).strip()
    return s.split() if s else []

def get_source_text_task1(row: pd.Series) -> str:
    # Para Task 1 usamos unicamente el texto del articulo
    return str(row.get("article_body", "") or "").strip()

def get_titles(row: pd.Series) -> List[str]:
    return [str(row.get(c, "") or "") for c in TITLE_COLS]

def _parse_rank_list(x: Any) -> List[str]:
    # Acepta formatos con espacios, comas o lista JSON
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    s = str(x).strip()
    if not s:
        return []

    if s.startswith("[") and s.endswith("]"):
        try:
            arr = json.loads(s)
            if isinstance(arr, list):
                return [str(t).strip() for t in arr if str(t).strip()]
        except Exception:
            pass

    s = s.replace("\t", " ").replace("\n", " ").replace("\r", " ").replace(";", " ")
    if "," in s:
        parts = [p.strip() for p in s.split(",")]
    else:
        parts = [p.strip() for p in s.split()]
    return [p for p in parts if p]

def _token_to_col(tok: Any) -> Optional[int]:
    if tok is None or (isinstance(tok, float) and pd.isna(tok)):
        return None
    s = str(tok).strip().lower()
    if len(s) < 2 or s[0] not in ("t", "d"):
        return None
    try:
        return int(s[1:])
    except Exception:
        return None

def _unique_valid_pred_cols(pred: List[str], n_cols: int) -> List[int]:
    out: List[int] = []
    seen = set()
    for tok in pred:
        c = _token_to_col(tok)
        if c is None or not (1 <= c <= n_cols):
            continue
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

def _ndcg_from_ideal(pred_cols: List[int], ideal_cols: List[int], k: int) -> float:
    if not ideal_cols:
        return 0.0

    ideal_pos = {c: i + 1 for i, c in enumerate(ideal_cols)}

    def gain(c: int) -> float:
        pos = ideal_pos.get(c, None)
        if pos is None:
            return 0.0
        return float(len(ideal_cols) - pos + 1)

    dcg = 0.0
    for i, c in enumerate(pred_cols[:k], start=1):
        dcg += gain(c) / math.log2(i + 1)

    idcg = 0.0
    for i, c in enumerate(ideal_cols[:k], start=1):
        idcg += gain(c) / math.log2(i + 1)

    if idcg <= 0:
        return 0.0
    return max(0.0, min(1.0, dcg / idcg))

def pa_ndcg(pred_tokens: List[str], true_tokens: List[str], k: int = 10, alpha: float = 0.9) -> float:
    # Metrica oficial del reto: top-1 + ranking global
    y = _unique_valid_pred_cols(true_tokens, N_COLS)
    p = _unique_valid_pred_cols(pred_tokens, N_COLS)

    if not y:
        return 0.0

    top1 = 1.0 if (p and p[0] == y[0]) else 0.0

    pred_rest = p[1:]
    ideal_rest = y[1:]
    aux = _ndcg_from_ideal(pred_rest, ideal_rest, k=k)

    return float(alpha * top1 + (1.0 - alpha) * aux)

def _dcg_from_rels(rels: List[float], k: int) -> float:
    dcg = 0.0
    for i, rel in enumerate(rels[:k], start=1):
        dcg += float(rel) / math.log2(i + 1)
    return dcg

def ndcg_standard(pred_tokens: List[str], true_tokens: List[str], k: int = 10) -> float:
    # NDCG estandar para comparar calidad de ranking completa
    ideal_cols = _unique_valid_pred_cols(true_tokens, N_COLS)
    pred_cols = _unique_valid_pred_cols(pred_tokens, N_COLS)
    if not ideal_cols:
        return 0.0

    ideal_rank = {c: i for i, c in enumerate(ideal_cols)}

    def rel(c: int) -> float:
        r = ideal_rank.get(c, None)
        return float(len(ideal_cols) - r) if r is not None else 0.0

    pred_rels = [rel(c) for c in pred_cols]
    ideal_rels = [rel(c) for c in ideal_cols]

    dcg = _dcg_from_rels(pred_rels, k)
    idcg = _dcg_from_rels(ideal_rels, k)
    return 0.0 if idcg <= 0 else max(0.0, min(1.0, dcg / idcg))

def mrr_at_k(pred_tokens: List[str], true_tokens: List[str], k: int = 10) -> float:
    y = _unique_valid_pred_cols(true_tokens, N_COLS)
    p = _unique_valid_pred_cols(pred_tokens, N_COLS)
    if not y or not p:
        return 0.0

    target = y[0]
    for i, c in enumerate(p[:k], start=1):
        if c == target:
            return 1.0 / float(i)
    return 0.0

def precision_at_k(pred_tokens: List[str], true_tokens: List[str], k: int) -> float:
    y = set(_unique_valid_pred_cols(true_tokens, N_COLS)[:k])
    p = _unique_valid_pred_cols(pred_tokens, N_COLS)[:k]
    if k <= 0:
        return 0.0
    if not p:
        return 0.0
    hits = sum(1 for c in p if c in y)
    return float(hits) / float(k)

def score_submission_task1_df(df_eval: pd.DataFrame, k: int = 10, alpha: float = 0.9) -> Dict[str, float]:
    n_total = len(df_eval)
    coverage = float(df_eval["task_1"].notna().mean()) if n_total else 0.0

    scores: List[float] = []
    for _, r in df_eval.iterrows():
        y = _parse_rank_list(r.get("y_true"))
        p = _parse_rank_list(r.get("task_1"))
        scores.append(pa_ndcg(p, y, k=k, alpha=alpha) if p else 0.0)

    task_1_score = float(np.mean(scores)) if scores else 0.0
    return {"task_1_pa_ndcg": task_1_score, "coverage": coverage, "k": k, "alpha": alpha}


# 5. Limpieza de texto y señal de word overlap

Esta parte prepara una señal lógica simple basada en la coincidencia entre palabras del título y del cuerpo del artículo.


In [17]:
# Limpieza simple de texto y calculo de word overlap

def clean_text(text: Any) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    s = str(text).lower()
    s = re.sub(r"\d+", " ", s)
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s).strip()
    return s


def overlap_score(title: str, body: str) -> float:
    # Ratio de palabras del titulo que aparecen en el cuerpo
    title_words = set(clean_text(title).split())
    body_words = set(clean_text(body).split())

    if not title_words:
        return 0.0
    return len(title_words.intersection(body_words)) / float(len(title_words))


def _minmax_01(x: np.ndarray) -> np.ndarray:
    # Normalizacion por fila para poder mezclar señales en escalas comparables
    x = x.astype(float)
    mn, mx = float(np.min(x)), float(np.max(x))
    if mx - mn < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - mn) / (mx - mn)


def _normalize_rows_01(x: np.ndarray) -> np.ndarray:
    x = x.astype(float)
    mn = x.min(axis=1, keepdims=True)
    mx = x.max(axis=1, keepdims=True)
    denom = mx - mn
    out = np.zeros_like(x, dtype=float)
    valid = denom[:, 0] >= 1e-12
    out[valid] = (x[valid] - mn[valid]) / denom[valid]
    return out


# 6. Modelo híbrido y funciones de predicción

Aquí se define el ranker que combina tres señales:
- bi-encoder,
- TF-IDF,
- overlap.

Además se incluyen las funciones para precomputar componentes, predecir rankings y buscar los mejores pesos en `dev`.


In [18]:
class HybridBiEncoderTfidfOverlapRanker:
    # Fusiona bi-encoder, TF-IDF y overlap para rankear los 10 titulos de Task 1.

    def __init__(
        self,
        model_name: str,
        w_bi_encoder: float = 0.55,
        w_tfidf: float = 0.30,
        w_overlap: float = 0.15,
    ):
        self.model_name = model_name
        self.w_bi_encoder = w_bi_encoder
        self.w_tfidf = w_tfidf
        self.w_overlap = w_overlap
        self.encoder: SentenceTransformer | None = None
        self.vectorizer: TfidfVectorizer | None = None

    def fit(self, train_df: pd.DataFrame) -> None:
        # Entrenamos TF-IDF con texto de articulos + candidatos de train
        sources = train_df["article_body"].fillna("").astype(str).tolist()
        titles_flat: List[str] = []
        for col in TITLE_COLS:
            titles_flat.extend(train_df[col].fillna("").astype(str).tolist())

        corpus = sources + titles_flat
        stopwords_es = stopwords.words("spanish")

        self.vectorizer = TfidfVectorizer(
            max_features=50_000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.9,
            sublinear_tf=True,
            stop_words=stopwords_es,
            norm="l2",
        )
        self.vectorizer.fit(corpus)

        if self.w_bi_encoder <= 0:
            self.encoder = None
            print("Carga del bi-encoder omitida porque su peso es 0")
            return

        model_path = resolve_model_path(
            model_name=self.model_name,
            local_models_dir=LOCAL_MODELS_DIR,
            download_if_missing=DOWNLOAD_MODEL_IF_MISSING,
        )
        self.encoder = SentenceTransformer(model_path)
        safe_max_seq_length = min(MODEL_MAX_SEQ_LENGTH, self.encoder.max_seq_length)
        self.encoder.max_seq_length = safe_max_seq_length
        if hasattr(self.encoder, "tokenizer") and hasattr(self.encoder.tokenizer, "model_max_length"):
            self.encoder.tokenizer.model_max_length = safe_max_seq_length

    def score_components(
        self,
        source_text: str,
        titles: List[str],
        weights: Optional[Dict[str, float]] = None,
    ) -> Dict[str, np.ndarray]:
        if self.vectorizer is None:
            raise RuntimeError("El ranker no esta entrenado")

        if weights is None:
            weights = _weights_from_values(self.w_bi_encoder, self.w_tfidf, self.w_overlap)

        active_keys = set(_active_signal_keys(weights))
        zero_scores = np.zeros(len(titles), dtype=float)
        components: Dict[str, np.ndarray] = {}

        if "tfidf" in active_keys:
            src_vec = self.vectorizer.transform([source_text])
            title_vecs = self.vectorizer.transform(titles)
            sims_tfidf = (title_vecs @ src_vec.T).toarray().ravel()
            components["tfidf"] = _minmax_01(sims_tfidf)
        else:
            components["tfidf"] = zero_scores.copy()

        if "bi_encoder" in active_keys:
            if self.encoder is None:
                raise RuntimeError("El encoder no esta cargado")
            texts = [source_text] + [str(t or "") for t in titles]
            embs = self.encoder.encode(texts, normalize_embeddings=True, convert_to_numpy=True)
            src_emb = embs[0]
            title_embs = embs[1:]
            sims_bi = np.dot(title_embs, src_emb)
            components["bi_encoder"] = _minmax_01(sims_bi)
        else:
            components["bi_encoder"] = zero_scores.copy()

        if "overlap" in active_keys:
            sims_overlap = np.array([overlap_score(t, source_text) for t in titles], dtype=float)
            components["overlap"] = _minmax_01(sims_overlap)
        else:
            components["overlap"] = zero_scores.copy()

        return components

    @staticmethod
    def rank_from_components(components: Dict[str, np.ndarray], weights: Dict[str, float]) -> List[int]:
        final_scores = _combine_weighted_components(components, weights)

        order = np.argsort(-final_scores)
        return order.tolist()

    def rank_titles(
        self,
        source_text: str,
        titles: List[str],
        weights: Optional[Dict[str, float]] = None,
    ) -> List[int]:
        if weights is None:
            weights = _weights_from_values(self.w_bi_encoder, self.w_tfidf, self.w_overlap)
        components = self.score_components(source_text, titles, weights=weights)
        return self.rank_from_components(components, weights)


def _extract_sources_and_titles(df_pred: pd.DataFrame) -> tuple[List[str], List[List[str]], List[str]]:
    sources = df_pred["article_body"].fillna("").astype(str).tolist()
    title_columns = [df_pred[c].fillna("").astype(str).tolist() for c in TITLE_COLS]
    titles_by_row = [list(items) for items in zip(*title_columns)]
    flat_titles = [title for row in titles_by_row for title in row]
    return sources, titles_by_row, flat_titles


def _compute_tfidf_matrix(
    sources: List[str],
    flat_titles: List[str],
    ranker: HybridBiEncoderTfidfOverlapRanker,
) -> np.ndarray:
    if ranker.vectorizer is None:
        raise RuntimeError("El vectorizador TF-IDF no esta entrenado")

    src_vecs = ranker.vectorizer.transform(sources)
    title_vecs = ranker.vectorizer.transform(flat_titles)

    n_rows = len(sources)
    sims_tfidf = np.zeros((n_rows, N_COLS), dtype=float)
    for i in range(n_rows):
        block = title_vecs[i * N_COLS : (i + 1) * N_COLS]
        sims_tfidf[i] = (block @ src_vecs[i].T).toarray().ravel()
    return _normalize_rows_01(sims_tfidf)


def _compute_bi_encoder_matrix(
    sources: List[str],
    flat_titles: List[str],
    ranker: HybridBiEncoderTfidfOverlapRanker,
) -> np.ndarray:
    if ranker.encoder is None:
        raise RuntimeError("El encoder no esta cargado")

    src_embs = ranker.encoder.encode(
        sources,
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=SHOW_PROGRESS_BAR,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    title_embs = ranker.encoder.encode(
        flat_titles,
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=SHOW_PROGRESS_BAR,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    n_rows = len(sources)
    title_embs = title_embs.reshape(n_rows, N_COLS, -1)
    sims_bi = np.einsum("nid,nd->ni", title_embs, src_embs, optimize=True)
    return _normalize_rows_01(sims_bi)


def _compute_overlap_matrix(sources: List[str], titles_by_row: List[List[str]]) -> np.ndarray:
    overlaps = np.zeros((len(sources), N_COLS), dtype=float)
    for i, (source_text, titles) in enumerate(zip(sources, titles_by_row)):
        cleaned_body_words = set(clean_text(source_text).split())
        row_scores: List[float] = []
        for title in titles:
            title_words = set(clean_text(title).split())
            if not title_words:
                row_scores.append(0.0)
            else:
                row_scores.append(len(title_words.intersection(cleaned_body_words)) / float(len(title_words)))
        overlaps[i] = row_scores
    return _normalize_rows_01(overlaps)


def _build_cached_rows_from_matrices(
    index: pd.Index,
    bi_matrix: np.ndarray,
    tfidf_matrix: np.ndarray,
    overlap_matrix: np.ndarray,
) -> List[Dict[str, Any]]:
    cached_rows: List[Dict[str, Any]] = []
    for idx, bi_row, tfidf_row, overlap_row in zip(index, bi_matrix, tfidf_matrix, overlap_matrix):
        cached_rows.append(
            {
                "index": idx,
                "components": {
                    "bi_encoder": bi_row,
                    "tfidf": tfidf_row,
                    "overlap": overlap_row,
                },
            }
        )
    return cached_rows


def predict_task1(
    df_pred: pd.DataFrame,
    ranker: HybridBiEncoderTfidfOverlapRanker,
    weights: Optional[Dict[str, float]] = None,
) -> pd.Series:
    preds: List[str] = []
    for _, row in df_pred.iterrows():
        source = get_source_text_task1(row)
        titles = get_titles(row)
        order = ranker.rank_titles(source, titles, weights=weights)
        ranked_tokens = [f"t{i+1}" for i in order]
        preds.append(" ".join(ranked_tokens))

    return pd.Series(preds, index=df_pred.index)


def precompute_components(
    df_pred: pd.DataFrame,
    ranker: HybridBiEncoderTfidfOverlapRanker,
    weights: Optional[Dict[str, float]] = None,
) -> List[Dict[str, Any]]:
    sources, titles_by_row, flat_titles = _extract_sources_and_titles(df_pred)
    n_rows = len(sources)

    if weights is None:
        weights = _weights_from_values(ranker.w_bi_encoder, ranker.w_tfidf, ranker.w_overlap)

    active_keys = set(_active_signal_keys(weights))
    bi_matrix = np.zeros((n_rows, N_COLS), dtype=float)
    tfidf_matrix = np.zeros((n_rows, N_COLS), dtype=float)
    overlap_matrix = np.zeros((n_rows, N_COLS), dtype=float)

    if "tfidf" in active_keys:
        t0 = time.perf_counter()
        tfidf_matrix = _compute_tfidf_matrix(sources, flat_titles, ranker)
        print(f"TF-IDF precomputado en {time.perf_counter() - t0:.2f}s")
    else:
        print("TF-IDF omitido porque su peso es 0")

    if "bi_encoder" in active_keys:
        t0 = time.perf_counter()
        bi_matrix = _compute_bi_encoder_matrix(sources, flat_titles, ranker)
        print(f"Bi-encoder precomputado en {time.perf_counter() - t0:.2f}s")
    else:
        print("Bi-encoder omitido porque su peso es 0")

    if "overlap" in active_keys:
        t0 = time.perf_counter()
        overlap_matrix = _compute_overlap_matrix(sources, titles_by_row)
        print(f"Overlap precomputado en {time.perf_counter() - t0:.2f}s")
    else:
        print("Overlap omitido porque su peso es 0")

    return _build_cached_rows_from_matrices(df_pred.index, bi_matrix, tfidf_matrix, overlap_matrix)


def _stack_component_matrix(cached_rows: List[Dict[str, Any]], key: str) -> np.ndarray:
    return np.vstack([row["components"][key] for row in cached_rows]).astype(float)


def _orders_to_prediction_strings(orders: np.ndarray) -> pd.Series:
    preds = [" ".join(f"t{idx}" for idx in row) for row in orders]
    return pd.Series(preds)


def predict_task1_from_cache(
    df_pred: pd.DataFrame,
    cached_rows: List[Dict[str, Any]],
    weights: Dict[str, float],
) -> pd.Series:
    component_matrices = {
        "bi_encoder": _stack_component_matrix(cached_rows, "bi_encoder"),
        "tfidf": _stack_component_matrix(cached_rows, "tfidf"),
        "overlap": _stack_component_matrix(cached_rows, "overlap"),
    }
    final_scores = _combine_weighted_components(component_matrices, weights)
    orders = np.argsort(-final_scores, axis=1) + 1
    preds = _orders_to_prediction_strings(orders)
    preds.index = df_pred.index
    return preds


def build_weight_grid(step: float = 0.05) -> List[Dict[str, float]]:
    if step <= 0 or step > 1:
        raise ValueError("step debe estar en (0, 1]")

    steps = int(round(1.0 / step))
    grid: List[Dict[str, float]] = []
    for bi_units in range(steps + 1):
        for tfidf_units in range(steps - bi_units + 1):
            overlap_units = steps - bi_units - tfidf_units
            grid.append(
                {
                    "bi_encoder": bi_units * step,
                    "tfidf": tfidf_units * step,
                    "overlap": overlap_units * step,
                }
            )
    return grid


def _pa_ndcg_from_cols(pred_cols: np.ndarray, true_tokens: List[str], k: int = 10, alpha: float = 0.9) -> float:
    pred_list = [f"t{int(c)}" for c in pred_cols.tolist()]
    return pa_ndcg(pred_list, true_tokens, k=k, alpha=alpha)


def search_best_weights(
    df_eval: pd.DataFrame,
    cached_rows: List[Dict[str, Any]],
    step: float = 0.05,
    base_weights: Optional[Dict[str, float]] = None,
) -> Dict[str, Any]:
    if "y_true" not in df_eval.columns:
        raise ValueError("Se requiere la columna y_true para buscar pesos")

    component_matrices = {
        "bi_encoder": _stack_component_matrix(cached_rows, "bi_encoder"),
        "tfidf": _stack_component_matrix(cached_rows, "tfidf"),
        "overlap": _stack_component_matrix(cached_rows, "overlap"),
    }
    true_tokens_by_row = [_parse_rank_list(v) for v in df_eval["y_true"].tolist()]
    disabled_signals = set()
    if base_weights is not None:
        active_keys = _active_signal_keys(base_weights)
        if not active_keys:
            raise ValueError("Se requiere al menos una señal activa para buscar pesos")
        disabled_signals = set(SIGNAL_KEYS) - set(active_keys)

    best_result: Dict[str, Any] | None = None
    for weights in build_weight_grid(step=step):
        if disabled_signals and any(weights[key] > 0 for key in disabled_signals):
            continue
        final_scores = _combine_weighted_components(component_matrices, weights)
        orders = np.argsort(-final_scores, axis=1) + 1

        row_scores = [
            _pa_ndcg_from_cols(pred_cols, true_tokens, k=NDCG_K, alpha=ALPHA)
            if true_tokens
            else 0.0
            for pred_cols, true_tokens in zip(orders, true_tokens_by_row)
        ]
        score = float(np.mean(row_scores)) if row_scores else 0.0

        if best_result is None or score > best_result["score"]:
            preds = _orders_to_prediction_strings(orders)
            preds.index = df_eval.index
            best_result = {
                "weights": weights,
                "score": score,
                "official": {
                    "task_1_pa_ndcg": score,
                    "coverage": 1.0 if len(df_eval) else 0.0,
                    "k": NDCG_K,
                    "alpha": ALPHA,
                },
                "predictions": preds,
            }

    if best_result is None:
        raise RuntimeError("No se pudo encontrar una combinacion valida de pesos")

    return best_result


# 7. Entrenamiento, precomputación y autoajuste de pesos

En este bloque se entrena el ranker, se calculan las señales sobre `dev` y, si existe `y_true`, se busca la mejor combinación de pesos automática.


In [19]:
# Entrenamos con train, precomputamos señales en dev y buscamos los mejores pesos
ranker = HybridBiEncoderTfidfOverlapRanker(
    model_name=MODEL_NAME,
    w_bi_encoder=W_BI_ENCODER,
    w_tfidf=W_TFIDF,
    w_overlap=W_OVERLAP,
)

t0 = time.perf_counter()
ranker.fit(train_df)
print(f"Entrenamiento y carga del modelo completados en {time.perf_counter() - t0:.2f}s")

selected_weights = _weights_from_values(W_BI_ENCODER, W_TFIDF, W_OVERLAP)

t0 = time.perf_counter()
cached_dev_rows = precompute_components(dev_df, ranker, weights=selected_weights)
print(f"Precomputacion total completada en {time.perf_counter() - t0:.2f}s")

autotune_info = {"enabled": False, "grid_step": WEIGHT_GRID_STEP}

if AUTO_TUNE_WEIGHTS and "y_true" in dev_df.columns:
    t0 = time.perf_counter()
    best_result = search_best_weights(
        dev_df,
        cached_dev_rows,
        step=WEIGHT_GRID_STEP,
        base_weights=selected_weights,
    )
    print(f"Autoajuste de pesos completado en {time.perf_counter() - t0:.2f}s")
    selected_weights = best_result["weights"]
    dev_df["task_1"] = best_result["predictions"]
    autotune_info = {
        "enabled": True,
        "grid_step": WEIGHT_GRID_STEP,
        "best_internal_pa_ndcg": float(best_result["score"]),
    }
else:
    dev_df["task_1"] = predict_task1_from_cache(dev_df, cached_dev_rows, selected_weights)

ranker.w_bi_encoder = selected_weights["bi_encoder"]
ranker.w_tfidf = selected_weights["tfidf"]
ranker.w_overlap = selected_weights["overlap"]

print("Pesos seleccionados:", selected_weights)
dev_df[["id", "task_1"]].head(5)


Usando modelo local: models\paraphrase-multilingual-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2197.06it/s]
XLMRobertaModel LOAD REPORT from: models\paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Entrenamiento y carga del modelo completados en 53.33s
TF-IDF omitido porque su peso es 0


Batches: 100%|██████████| 768/768 [25:21<00:00,  1.98s/it]


Bi-encoder precomputado en 2205.84s
Overlap omitido porque su peso es 0
Precomputacion total completada en 2205.91s
Pesos seleccionados: {'bi_encoder': 1, 'tfidf': 0, 'overlap': 0}


,id,task_1
0,53220bb11b7b235667e56fae56abbc9a2a3269fdfa5188...,t2 t8 t4 t10 t3 t1 t9 t7 t6 t5
1,8bf0cb56c3262c158173e10d43e6aaee683569a0d5a6dd...,t9 t2 t5 t6 t1 t10 t8 t3 t7 t4
2,e973b790bf855b8cb275a7645616770e38eaa48bb90016...,t9 t2 t3 t1 t4 t8 t10 t7 t6 t5
3,9e7e2827a53f14e44d36becdbe1ac5119752587812bb4f...,t8 t10 t3 t6 t9 t7 t1 t4 t2 t5
4,041221e337a91d336cf2c5f42153f608ef87ca9fa15714...,t3 t10 t9 t6 t4 t7 t8 t2 t5 t1


# 8. Exportación de la submission

Se guarda el archivo final en el formato esperado por la competición. `task_2` replica `task_1` por compatibilidad con el formato oficial.


In [20]:
# Guardamos submission con formato oficial (task_2 replica task_1)
# Nota: aunque no participemos en Task 2, se mantiene la columna por compatibilidad.
submission = dev_df[["id", "task_1"]].copy()
submission["task_2"] = submission["task_1"]
submission = submission[["id", "task_1", "task_2"]]
submission.to_csv(OUTPUT_SUBMISSION, index=False)
print("Guardado:", OUTPUT_SUBMISSION)
submission.head(5)


Guardado: results_evaluation.csv


,id,task_1,task_2
0,53220bb11b7b235667e56fae56abbc9a2a3269fdfa5188...,t2 t8 t4 t10 t3 t1 t9 t7 t6 t5,t2 t8 t4 t10 t3 t1 t9 t7 t6 t5
1,8bf0cb56c3262c158173e10d43e6aaee683569a0d5a6dd...,t9 t2 t5 t6 t1 t10 t8 t3 t7 t4,t9 t2 t5 t6 t1 t10 t8 t3 t7 t4
2,e973b790bf855b8cb275a7645616770e38eaa48bb90016...,t9 t2 t3 t1 t4 t8 t10 t7 t6 t5,t9 t2 t3 t1 t4 t8 t10 t7 t6 t5
3,9e7e2827a53f14e44d36becdbe1ac5119752587812bb4f...,t8 t10 t3 t6 t9 t7 t1 t4 t2 t5,t8 t10 t3 t6 t9 t7 t1 t4 t2 t5
4,041221e337a91d336cf2c5f42153f608ef87ca9fa15714...,t3 t10 t9 t6 t4 t7 t8 t2 t5 t1,t3 t10 t9 t6 t4 t7 t8 t2 t5 t1


# 9. Evaluación ampliada

Además de la métrica oficial, este bloque guarda métricas de análisis como `top1_accuracy`, `NDCG`, `MRR` y `precision@k`.


In [21]:
# Evaluacion ampliada de Task 1 y guardado de metricas del mejor resultado
if "y_true" in dev_df.columns:
    official = score_submission_task1_df(dev_df, k=NDCG_K, alpha=ALPHA)

    top1 = []
    pa = []
    nd = []
    mrr = []
    p1 = []
    p3 = []
    p5 = []

    for _, row in dev_df.iterrows():
        y = _parse_rank_list(row.get("y_true"))
        p = _parse_rank_list(row.get("task_1"))

        if not y:
            continue

        top1.append(1.0 if (p and p[0] == y[0]) else 0.0)
        pa.append(pa_ndcg(p, y, k=NDCG_K, alpha=ALPHA))
        nd.append(ndcg_standard(p, y, k=NDCG_K))
        mrr.append(mrr_at_k(p, y, k=NDCG_K))
        p1.append(precision_at_k(p, y, k=1))
        p3.append(precision_at_k(p, y, k=3))
        p5.append(precision_at_k(p, y, k=5))

    extra = {
        "top1_accuracy": float(np.mean(top1)) if top1 else 0.0,
        "pa_ndcg": float(np.mean(pa)) if pa else 0.0,
        "ndcg_standard": float(np.mean(nd)) if nd else 0.0,
        "mrr@10": float(np.mean(mrr)) if mrr else 0.0,
        "precision@1": float(np.mean(p1)) if p1 else 0.0,
        "precision@3": float(np.mean(p3)) if p3 else 0.0,
        "precision@5": float(np.mean(p5)) if p5 else 0.0,
    }

    print("Metricas oficiales:", official)
    print("Metricas extra (analisis):", extra)

    if SAVE_METRICS_JSON:
        metrics = {
            "method": METHOD,
            "model": MODEL_NAME,
            "weights": {
                "w_bi_encoder": float(selected_weights["bi_encoder"]),
                "w_tfidf": float(selected_weights["tfidf"]),
                "w_overlap": float(selected_weights["overlap"]),
            },
            "autotune": autotune_info,
            "n_train": int(len(train_df)),
            "n_dev": int(len(dev_df)),
            "metrics": {"official": official, "extra": extra},
        }
        metrics_path = Path(OUTPUT_SUBMISSION).with_suffix(".metrics.json")
        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2, ensure_ascii=False)
        print("Guardado:", metrics_path)
else:
    print("No se encontro y_true en dev; se omiten metricas.")


No se encontro y_true en dev; se omiten metricas.


# 10. Revisión rápida del Top-1 en las primeras noticias

Este último bloque resume cuántas veces el primer título predicho coincide con el correcto en las primeras `N` noticias.


In [22]:
# Modulo: aciertos del primer titulo en las primeras N noticias (por defecto, 50)

def top1_hits_first_n(df: pd.DataFrame, n: int = 50, pred_col: str = "task_1", true_col: str = "y_true") -> Dict[str, float]:
    if pred_col not in df.columns or true_col not in df.columns:
        return {"evaluated": 0, "hits_top1": 0, "accuracy_top1": 0.0}

    sub = df.head(n).copy()
    hits = 0
    evaluated = 0

    for _, row in sub.iterrows():
        y = _parse_rank_list(row.get(true_col))
        p = _parse_rank_list(row.get(pred_col))
        if not y or not p:
            continue
        evaluated += 1
        if p[0] == y[0]:
            hits += 1

    acc = (hits / evaluated) if evaluated else 0.0
    return {"evaluated": int(evaluated), "hits_top1": int(hits), "accuracy_top1": float(acc)}

if "y_true" in dev_df.columns and "task_1" in dev_df.columns:
    top1_50 = top1_hits_first_n(dev_df, n=50)
    print(f"Top-1 en primeras 50 noticias: {top1_50['hits_top1']} / {top1_50['evaluated']} (acc={top1_50['accuracy_top1']:.4f})")
else:
    print("No se puede calcular Top-1: faltan columnas 'task_1' o 'y_true'.")


No se puede calcular Top-1: faltan columnas 'task_1' o 'y_true'.
